In [ ]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label", "narration", "mode", "type", "category", "subcategory"]]

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    df["input_text"],
    df["label"],
    df,   # 👈 IMPORTANT
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
from gliner import GLiNER

# GLiNER2 model
gliner_model = GLiNER.from_pretrained("urchade/gliner_large-v2")

labels = list(set(y_train))

In [ ]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        best = max(entities, key=lambda x: x["score"])
        return best["label"], best["score"]

    return "unknown_unknown", 0.0

In [ ]:
y_pred = []
conf_scores = []

for text in X_test:
    entities_batch = gliner_model.predict_entities(list(X_test), labels)

In [ ]:
result_df = df_test.copy()

result_df["predicted_label"] = y_pred
result_df["confidence"] = conf_scores

In [ ]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.6:
        return "High"
    elif c >= 0.3:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [ ]:
# Actual split
result_df[["category", "subcategory"]] = result_df["actual_label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [ ]:
result_df["correct"] = result_df["actual_label"] == result_df["predicted_label"]

In [ ]:
print(result_df.columns)

In [ ]:
final_df = result_df[
    [
        "narration",
        "mode",
        "type",
        "category",
        "subcategory",
        "predicted_category",
        "predicted_subcategory",
        "confidence",
        "confidence_percent",
        "confidence_level",
        "correct"
    ]
]

final_df.head(20)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# F2 Score
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print("\nClassification Report:")
print(report_df)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])

In [ ]:
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])